In [27]:
import pathlib

users_line = next(l for l in pathlib.Path('run.sh').read_text().splitlines() if l.startswith('USERS='))
users_str  = users_line.split('=', 1)[1].strip('"\'')

sender_ap, recipient_ap = users_str.split(',')

def ap(c):
  up, d = c.split('@')
  u, p = up.split(':')
  return f'{u}@{d}', p

sender, sender_pass = ap(sender_ap)
recipient, recipient_pass = ap(recipient_ap)

In [28]:
import smtplib
from email.message import EmailMessage

msg = EmailMessage()
msg['Subject'] = 'Test message'
msg['From'] = sender
msg['To'] = recipient
msg.set_content('Hello from GreenMail!')

with smtplib.SMTP('localhost', 3025) as smtp:
    smtp.login(sender, sender_pass)
    smtp.send_message(msg)

print('Email sent.')

Email sent.


In [29]:
import imaplib
import email

with imaplib.IMAP4('localhost', 3143) as imap:
    imap.login(recipient, recipient_pass)
    imap.select('INBOX')
    _, msg_nums = imap.search(None, 'ALL')
    for num in msg_nums[0].split():
        _, data = imap.fetch(num, '(RFC822)')
        msg = email.message_from_bytes(data[0][1])
        print(f"From:    {msg['From']}")
        print(f"Subject: {msg['Subject']}")
        print(f"Body:    {msg.get_payload()}")

From:    snd@foo.bar
Subject: Test message
Body:    Hello from GreenMail!
From:    snd@foo.bar
Subject: Results
Body:    [<email.message.Message object at 0x7af6602a5af0>, <email.message.Message object at 0x7af6602a6180>]
From:    snd@foo.bar
Subject: Test message
Body:    Hello from GreenMail!


In [30]:
from imap_tools import MailBoxUnencrypted

with MailBoxUnencrypted('localhost', 3143).login(recipient, recipient_pass) as mailbox:
    for msg in mailbox.fetch():
        print(f"From:    {msg.from_}")
        print(f"Subject: {msg.subject}")
        print(f"Body:    {msg.text}")

From:    snd@foo.bar
Subject: Test message
Body:    Hello from GreenMail!
From:    snd@foo.bar
Subject: Results
Body:    Please find the results attached.

From:    snd@foo.bar
Subject: Test message
Body:    Hello from GreenMail!


## Attachment handling: `imaplib` vs `imap_tools`

Send a message with a CSV attachment, then retrieve it two ways.

In [31]:
import smtplib
from email.message import EmailMessage

csv_data = "name,score\nAlice,95\nBob,87\nCarol,92\n"

msg = EmailMessage()
msg['Subject'] = 'Results'
msg['From'] = sender
msg['To'] = recipient
msg.set_content('Please find the results attached.')
msg.add_attachment(csv_data.encode(), maintype='text', subtype='csv', filename='results.csv')

with smtplib.SMTP('localhost', 3025) as smtp:
    smtp.login(sender, sender_pass)
    smtp.send_message(msg)

print('Email with attachment sent.')

Email with attachment sent.


In [32]:
import imaplib, email as emaillib

with imaplib.IMAP4('localhost', 3143) as imap:
    imap.login(recipient, recipient_pass)
    imap.select('INBOX')
    _, msg_nums = imap.search(None, 'ALL')
    _, data = imap.fetch(msg_nums[0].split()[-1], '(RFC822)')

msg = emaillib.message_from_bytes(data[0][1])
for part in msg.walk():
    if part.get_content_disposition() == 'attachment':
        print(f"Filename: {part.get_filename()}")
        print(part.get_payload(decode=True).decode())

Filename: results.csv
name,score
Alice,95
Bob,87
Carol,92



In [33]:
from imap_tools import MailBoxUnencrypted

with MailBoxUnencrypted('localhost', 3143).login(recipient, recipient_pass) as mailbox:
    msg = list(mailbox.fetch())[-1]

att = msg.attachments[0]
print(f"Filename: {att.filename}")
print(att.payload.decode())

Filename: results.csv
name,score
Alice,95
Bob,87
Carol,92

